# Step 1/2 — Discontinuity-aware extraction, silence removal, and windowing

**GENERIC TEMPLATE -- READ BEFORE USE.** This notebook is sex- and window-length-agnostic: it
loops over all three window lengths (`WINDOW_LENGTHS`) and all four sex x DEP/HEALTH groups
(`grupos`) in a single run. Before running it, edit `daic_dir` and `split_data_dir` in the
configuration cell of Step 1 to point at your local DAIC-WOZ copy and your desired output
folder, and edit `qc_subject_ids` in the QC cell to subjects you know well.

**Motivation.** Ellie's turns are removed from the transcript, and the remaining Participant
fragments are treated as a sequence of temporally distinct spans rather than concatenated into
one continuous blob before windowing. Silences >=0.5s are removed programmatically, and every
cut is logged. Two temporally-adjacent windows can still come from two originally
non-contiguous spans of the recording -- separated by an Ellie turn, a long pause, or a silence
cut -- and that splice is not a natural acoustic transition. Downstream, `stability(k)` (cosine
similarity between window *t* and *t-1*) and the delta ($\Delta X$) aggregation statistic are
both "temporal dynamics" features, so their computation must reset at every such splice, not
only at the start of each subject's data.

**What this notebook does.**
1. Extracts Participant audio via the transcript, without concatenating non-contiguous spans
   into one blob before windowing.
2. Removes silence programmatically (short-time RMS energy, no `librosa` dependency needed),
   using the >=0.5s duration rule described in the paper, so every cut is logged.
3. Tracks every surviving contiguous span as a **chunk**, with its own `chunk_id` and its true
   `[orig_start_s, orig_stop_s)` in the original recording.
4. Windows **each chunk independently** -- no window ever straddles a chunk boundary, and the
   remainder shorter than a full window is discarded per-chunk.
5. Saves, alongside each subject's windows array, a window-level metadata table whose
   `is_chunk_start` column tells the downstream VMD/feature-extraction notebooks exactly where
   `stability`/$\Delta X$ must be reset (the same convention used for the first window of each
   subject, now applied at every genuine discontinuity).

This notebook does **not** touch VMD, the Gaussian-kernel mode selection, or feature
aggregation -- those live in the six `ST_VMD_GK_*` notebooks. Its only job is to produce a
set of per-subject window arrays + metadata for them to consume. Wiring the six notebooks'
`stability`/$\Delta X$ computation to read `is_chunk_start` is a separate step, once this
preprocessing has been run and sanity-checked.

## Step 1 — Configuration

Sets the paths to the local DAIC-WOZ copy and the output folder, the sample rate, the three
window lengths evaluated in the paper, the programmatic silence-removal parameters, and the
TRAIN/TEST split settings. Also creates the output directory tree for every window-length x
split x group combination.

In [ ]:
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIGURATION
# ============================================================
daic_dir       = Path(r"D:\DAICWOZ")               # EDIT to your local DAIC-WOZ copy
split_data_dir = Path(r"D:\DAICWOZ_split_DATA")   # EDIT to your local output folder

SAMPLE_RATE     = 16000
WINDOW_LENGTHS  = [0.5, 1.0, 2.0]   # seconds -- matches the 3 window-length variants
                                    # already evaluated in the paper (0.5s / 1s / 2s)

# Silence-removal parameters (short-time RMS energy based).
# MIN_SILENCE_S matches the >=0.5s duration rule described in the paper.
# TOP_DB / FRAME_S / HOP_S should be sanity-checked (Cell "QC" below) against a
# couple of subjects you know well, adjusting TOP_DB if it over- or under-detects
# silence.
MIN_SILENCE_S = 0.5   # minimum gap duration to count as a removable silence
TOP_DB        = 30.0  # a frame is "silence" if it is TOP_DB below the subject's peak
FRAME_S       = 0.02  # 20ms analysis frame for the short-time RMS energy
HOP_S         = 0.01  # 10ms hop between frames

TEST_SIZE = 0.2
SEED      = 42

grupos = ["FEMALE_DEP", "FEMALE_HEALTH", "MALE_DEP", "MALE_HEALTH"]

for window_s in WINDOW_LENGTHS:
    tag = f"windows_{window_s}s".replace(".0s", "s")
    for split_name in ["TRAIN", "TEST"]:
        for grupo in grupos:
            (split_data_dir / tag / f"{split_name}_{grupo}").mkdir(parents=True, exist_ok=True)

print("Output root:", split_data_dir)

## Step 2 — Load subject metadata

Loads and merges the DAIC-WOZ split CSVs (TRAIN, DEV, and the labeled full TEST release) into
one `metadata` table with `Participant_ID`, `PHQ8_Score`, and `Gender`, handling the small
column-naming/separator differences between files.

In [ ]:
# ============================================================
# METADATA LOADING
# ============================================================
def load_metadata(daic_dir):
    split_files = [
        "train_split_Depression_AVEC2017.csv",
        "dev_split_Depression_AVEC2017.csv",
        "full_test_split.csv",   # released after AVEC2017 with PHQ_Score/Gender
                                  # (unlike test_split_Depression_AVEC2017.csv,
                                  # the original blind-challenge file, which has
                                  # no labels and is intentionally NOT read here)
    ]
    frames = []
    for split_file in split_files:
        path = daic_dir / split_file
        if not path.exists():
            print(f"WARNING - not found: {split_file}")
            continue
        try:
            df = pd.read_csv(path, sep=';', skip_blank_lines=True)
            col_map = {}
            for col in df.columns:
                cl = col.strip().lower()
                if 'participant' in cl:
                    col_map['Participant_ID'] = col
                elif 'phq8_score' in cl or 'phq_score' in cl:
                    col_map['PHQ8_Score'] = col
                elif cl == 'gender':
                    col_map['Gender'] = col
            if len(col_map) < 3:
                df = pd.read_csv(path, sep=',', skip_blank_lines=True)
                for col in df.columns:
                    cl = col.strip().lower()
                    if 'participant' in cl:
                        col_map['Participant_ID'] = col
                    elif 'phq8_score' in cl or 'phq_score' in cl:
                        col_map['PHQ8_Score'] = col
                    elif cl == 'gender':
                        col_map['Gender'] = col
            df = df.rename(columns={v: k for k, v in col_map.items()})
            df = df[['Participant_ID', 'PHQ8_Score', 'Gender']].copy()
            df['split'] = split_file.split('_')[0]
            df = df.dropna(subset=['PHQ8_Score', 'Gender'])
            frames.append(df)
            print(f"OK {split_file}: {len(df)} subjects")
        except Exception as e:
            print(f"ERROR reading {split_file}: {e}")

    metadata = pd.concat(frames, ignore_index=True)
    metadata = metadata.drop_duplicates(subset='Participant_ID')
    metadata['Participant_ID'] = metadata['Participant_ID'].astype(int)
    metadata['PHQ8_Score']     = metadata['PHQ8_Score'].astype(float)
    metadata['Gender']         = metadata['Gender'].astype(int)
    return metadata

metadata = load_metadata(daic_dir)
print(f"\nTotal subjects: {len(metadata)}")

## Step 3 — Discontinuity-aware chunking and windowing (core functions)

Defines the functions that detect silence programmatically and split each subject's audio into
contiguous **chunks** (bounded by Ellie turns and by internal silences >= 0.5s), then window
each chunk independently so no window ever straddles a discontinuity. `windows_from_chunks`
also produces the `is_chunk_start` metadata that downstream notebooks use to reset
`stability`/delta at genuine discontinuities.

In [ ]:
# ============================================================
# DISCONTINUITY-AWARE CORE
# ============================================================
def short_time_rms(x, frame, hop):
    """Vectorized short-time RMS via a strided (zero-copy) view."""
    n = len(x)
    if n == 0:
        return np.empty(0, dtype=np.float64)
    if n < frame:
        return np.array([np.sqrt(np.mean(x.astype(np.float64) ** 2))])
    n_frames = 1 + (n - frame) // hop
    x64 = np.ascontiguousarray(x, dtype=np.float64)
    shape = (n_frames, frame)
    strides = (x64.strides[0] * hop, x64.strides[0])
    frames = np.lib.stride_tricks.as_strided(x64, shape=shape, strides=strides,
                                              writeable=False)
    return np.sqrt(np.mean(frames ** 2, axis=1))


def detect_nonsilent_intervals(seg, sr, top_db=30.0, frame_s=0.02, hop_s=0.01,
                                ref_rms=None):
    """Returns [start_sample, end_sample) intervals of `seg` that are NOT
    silence, i.e. whose short-time RMS is within top_db dB of the peak."""
    frame = max(1, int(round(frame_s * sr)))
    hop   = max(1, int(round(hop_s * sr)))
    n = len(seg)
    if n == 0:
        return []
    rms = short_time_rms(seg, frame, hop)
    peak = ref_rms if ref_rms is not None else rms.max()
    if peak <= 0:
        return []
    ref_db   = 20.0 * np.log10(peak + 1e-12)
    frame_db = 20.0 * np.log10(rms + 1e-12)
    nonsilent = frame_db > (ref_db - top_db)

    intervals = []
    in_run = False
    run_start = 0
    for i, flag in enumerate(nonsilent):
        samp = i * hop
        if flag and not in_run:
            in_run = True
            run_start = samp
        elif not flag and in_run:
            in_run = False
            intervals.append((run_start, min(samp + frame, n)))
    if in_run:
        intervals.append((run_start, n))
    return intervals


def merge_short_gaps(intervals, sr, min_gap_s=0.5):
    """Only a gap >= min_gap_s counts as a genuine, removable silence --
    matches the paper's stated >0.5s removal rule."""
    if not intervals:
        return []
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        gap_s = (s - merged[-1][1]) / sr
        if gap_s < min_gap_s:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    return [tuple(iv) for iv in merged]


def extract_chunks_for_subject(subject_id, audio, sr, transcript,
                                min_silence_s=0.5, top_db=30.0,
                                frame_s=0.02, hop_s=0.01):
    """Ordered list of contiguous audio chunks for one subject. A boundary
    is introduced (a) between every pair of consecutive Participant rows
    (there is always at least an Ellie turn between them: 'turn change'),
    and (b) at every internal silence >= min_silence_s within a single
    Participant row (a long pause within that turn). No two chunks are
    ever treated as continuous.

    Each row's silence detector uses that row's OWN peak as its 0 dB
    reference (detect_nonsilent_intervals' default when ref_rms is not
    overridden), not one shared peak across the whole subject. A
    subject-wide reference would unfairly gut short, quiet turns (a
    one-word "yes" or "good") whenever the same interview also has a much
    louder, longer utterance: judged against that louder row's peak, the
    quiet-but-real row falls below top_db and is almost entirely discarded
    as silence. Judging each row against its own peak avoids that cross-row
    penalty while still correctly detecting genuine silence *within* that
    same row."""
    transcript = transcript.sort_values("start_time").reset_index(drop=True)
    participant_rows = transcript[transcript["speaker"] == "Participant"]

    raw_segments = []
    for _, row in participant_rows.iterrows():
        start = int(float(row["start_time"]) * sr)
        end   = min(int(float(row["stop_time"]) * sr), len(audio))
        if end > start:
            raw_segments.append((row, audio[start:end]))
    if not raw_segments:
        return []

    chunks = []
    chunk_counter = 0
    for row, seg in raw_segments:
        intervals = detect_nonsilent_intervals(
            seg, sr, top_db=top_db, frame_s=frame_s, hop_s=hop_s,
            ref_rms=None,   # per-row peak, not a subject-wide one -- see docstring
        )
        intervals = merge_short_gaps(intervals, sr, min_gap_s=min_silence_s)
        for s, e in intervals:
            if e <= s:
                continue
            chunks.append({
                "subject_id":   subject_id,
                "chunk_id":     chunk_counter,
                "orig_start_s": float(row["start_time"]) + s / sr,
                "orig_stop_s":  float(row["start_time"]) + e / sr,
                "audio":        seg[s:e],
            })
            chunk_counter += 1
    return chunks


def windows_from_chunks(chunks, window_s, sr):
    """Window each chunk independently -- no window crosses a chunk
    boundary; the remainder shorter than a full window is discarded
    per-chunk. `is_chunk_start=True` marks every window that starts a new
    chunk, i.e. every point where stability/delta must reset."""
    window_samples = int(round(window_s * sr))
    windows, meta = [], []
    for chunk in chunks:
        seg = chunk["audio"]
        n_here = len(seg) // window_samples
        for i in range(n_here):
            windows.append(seg[i * window_samples:(i + 1) * window_samples])
            meta.append({
                "subject_id":      chunk["subject_id"],
                "chunk_id":        chunk["chunk_id"],
                "window_in_chunk": i,
                "is_chunk_start":  i == 0,
                "orig_start_s":    chunk["orig_start_s"] + i * window_s,
                "orig_stop_s":     chunk["orig_start_s"] + (i + 1) * window_s,
            })
    if not windows:
        return (np.empty((0, window_samples), dtype=np.float32),
                pd.DataFrame(columns=["subject_id", "chunk_id", "window_in_chunk",
                                       "is_chunk_start", "orig_start_s", "orig_stop_s"]))
    return np.array(windows, dtype=np.float32), pd.DataFrame(meta)

print("Core functions loaded.")

## QC cell -- run this on 1-2 subjects you know well before the full loop

Reports, for a handful of subjects, how much audio survives the programmatic silence removal
and prints a couple of chunk boundaries so you can eyeball whether `TOP_DB` needs adjusting.
Raise `TOP_DB` if too much genuine speech is being cut as "silence"; lower it if obvious pauses
are being kept.

In [ ]:
# ============================================================
# QC -- inspect a couple of subjects before running the full loop
# ============================================================
qc_subject_ids = ["300", "305"]   # <-- edit to subjects you know well

for subject_id in qc_subject_ids:
    subj_dir = daic_dir / f"{subject_id}_P"
    audio_path = subj_dir / f"{subject_id}_AUDIO.wav"
    trans_path = subj_dir / f"{subject_id}_TRANSCRIPT.csv"
    if not audio_path.exists() or not trans_path.exists():
        print(f"{subject_id}: audio or transcript not found, skipping")
        continue

    audio, sr = sf.read(str(audio_path))
    transcript = pd.read_csv(trans_path, sep='\t')

    raw_participant_s = sum(
        min(float(r.stop_time), len(audio) / sr) - float(r.start_time)
        for _, r in transcript[transcript.speaker == "Participant"].iterrows()
    )

    chunks = extract_chunks_for_subject(subject_id, audio, sr, transcript,
                                         min_silence_s=MIN_SILENCE_S, top_db=TOP_DB,
                                         frame_s=FRAME_S, hop_s=HOP_S)
    kept_s = sum(len(c["audio"]) / sr for c in chunks)

    print(f"\nSubject {subject_id}")
    print(f"  Raw Participant audio (Ellie removed only): {raw_participant_s:.1f}s")
    print(f"  Kept after programmatic silence removal:    {kept_s:.1f}s "
          f"({100*kept_s/raw_participant_s:.0f}% kept)")
    print(f"  n_chunks: {len(chunks)}")
    print("  First 5 chunks (orig_start_s, orig_stop_s, dur_s):")
    for c in chunks[:5]:
        dur = c['orig_stop_s'] - c['orig_start_s']
        print(f"    chunk_id={c['chunk_id']:>3}  [{c['orig_start_s']:.2f}, "
              f"{c['orig_stop_s']:.2f}]  dur={dur:.2f}s")

## Step 4 — TRAIN/TEST split

Assigns each subject to TRAIN or TEST within its sex x DEP/HEALTH group, using a stratified
split with a fixed seed and proportions.

In [ ]:
# ============================================================
# TRAIN/TEST SPLIT (stratified by sex x DEP/HEALTH group, fixed seed)
# ============================================================
metadata['gender_str'] = metadata['Gender'].map({0: "FEMALE", 1: "MALE"})
metadata['dep_str']    = metadata['PHQ8_Score'].apply(lambda x: "DEP" if x >= 10 else "HEALTH")
metadata['grupo']      = metadata['gender_str'] + "_" + metadata['dep_str']

assignment = {}   # subject_id (str) -> (split_name, grupo)
for grupo in grupos:
    subj_ids = metadata.loc[metadata['grupo'] == grupo, 'Participant_ID'].astype(str).tolist()
    train_ids, test_ids = train_test_split(subj_ids, test_size=TEST_SIZE, random_state=SEED)
    for sid in train_ids:
        assignment[sid] = ("TRAIN", grupo)
    for sid in test_ids:
        assignment[sid] = ("TEST", grupo)
    print(f"{grupo}: {len(subj_ids)} subjects -> {len(train_ids)} TRAIN / {len(test_ids)} TEST")

print(f"\nTotal assigned: {len(assignment)}")

## Step 5 — Main loop

For every subject: extracts its discontinuity-aware chunks once, then windows those chunks at
each of the three window lengths, saving the windows array and the window-level metadata
(including `is_chunk_start`) to the corresponding TRAIN/TEST x group output folder. Also builds
the overall processing manifest.

In [ ]:
# ============================================================
# MAIN LOOP -- extract chunks ONCE per subject, then window at each of the
# three window lengths and save windows + metadata for each.
# ============================================================
manifest_rows = []

for subject_id, (split_name, grupo) in assignment.items():
    subj_dir   = daic_dir / f"{subject_id}_P"
    audio_path = subj_dir / f"{subject_id}_AUDIO.wav"
    trans_path = subj_dir / f"{subject_id}_TRANSCRIPT.csv"

    if not audio_path.exists() or not trans_path.exists():
        print(f"[{split_name}] {subject_id} -> MISSING audio/transcript, skipped")
        manifest_rows.append({"subject_id": subject_id, "split": split_name,
                               "grupo": grupo, "status": "missing_files"})
        continue

    audio, sr = sf.read(str(audio_path))
    if sr != SAMPLE_RATE:
        print(f"[{split_name}] {subject_id} -> unexpected SR={sr}, skipped")
        manifest_rows.append({"subject_id": subject_id, "split": split_name,
                               "grupo": grupo, "status": f"bad_sr_{sr}"})
        continue

    transcript = pd.read_csv(trans_path, sep='\t')

    chunks = extract_chunks_for_subject(subject_id, audio, sr, transcript,
                                         min_silence_s=MIN_SILENCE_S, top_db=TOP_DB,
                                         frame_s=FRAME_S, hop_s=HOP_S)
    if not chunks:
        print(f"[{split_name}] {subject_id} -> no chunks (no Participant audio), skipped")
        manifest_rows.append({"subject_id": subject_id, "split": split_name,
                               "grupo": grupo, "status": "no_chunks"})
        continue

    row_summary = {"subject_id": subject_id, "split": split_name, "grupo": grupo,
                    "status": "OK", "n_chunks": len(chunks)}

    for window_s in WINDOW_LENGTHS:
        tag = f"windows_{window_s}s".replace(".0s", "s")
        windows, meta = windows_from_chunks(chunks, window_s=window_s, sr=sr)

        dst_dir = split_data_dir / tag / f"{split_name}_{grupo}"
        np.save(str(dst_dir / f"{subject_id}_windows.npy"), windows)
        meta.to_csv(str(dst_dir / f"{subject_id}_window_meta.csv"), index=False)

        n_reset = int(meta['is_chunk_start'].sum()) if len(meta) else 0
        row_summary[f"n_windows_{window_s}s"] = len(windows)
        row_summary[f"n_resets_{window_s}s"]  = n_reset

    manifest_rows.append(row_summary)
    print(f"[{split_name}] {subject_id} ({grupo}) -> {len(chunks)} chunks, "
          f"windows: " + ", ".join(f"{w}s={row_summary.get(f'n_windows_{w}s', 0)}"
                                    for w in WINDOW_LENGTHS))

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(str(split_data_dir / "manifest.csv"), index=False)
print(f"\nManifest saved: {split_data_dir / 'manifest.csv'}")

## Step 6 — Summary

Reports how many subjects processed successfully, flags any with issues, and -- per window
length -- how many windows were produced in total and what fraction of them are discontinuity
resets (i.e. the fraction where `stability`/delta must reset because a genuine discontinuity
precedes that window).

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
ok = manifest_df[manifest_df['status'] == 'OK']
print(f"Subjects processed OK : {len(ok)} / {len(manifest_df)}")
if (manifest_df['status'] != 'OK').any():
    print("\nSubjects with issues:")
    print(manifest_df[manifest_df['status'] != 'OK'][['subject_id', 'grupo', 'status']]
          .to_string(index=False))

print("\nPer window-length totals:")
for window_s in WINDOW_LENGTHS:
    n_w = ok[f'n_windows_{window_s}s'].sum()
    n_r = ok[f'n_resets_{window_s}s'].sum()
    pct = 100 * n_r / n_w if n_w else 0
    print(f"  {window_s}s: {n_w} windows total, {n_r} are discontinuity resets "
          f"({pct:.1f}% of all windows -- i.e. this fraction gets stability=1 / "
          f"delta reset because a genuine discontinuity precedes that window)")

print("\nBy group x split:")
print(ok.groupby(['split', 'grupo']).agg(
    n_subjects=('subject_id', 'count'),
    n_chunks_mean=('n_chunks', 'mean'),
).round(1))